In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import csv
import pandas as pd
import scipy.io
import numpy as np
import copy

import matplotlib.pyplot as plt
import h5py
import scipy
from PIL import Image
from scipy import ndimage


from tensorflow import keras
import tensorflow as tf

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
!pip install watermark
%load_ext watermark


%watermark --iversions

PIL       : 11.2.1
pandas    : 2.2.2
numpy     : 2.0.2
csv       : 1.0
tensorflow: 2.18.0
h5py      : 3.14.0
keras     : 3.8.0
matplotlib: 3.10.0
scipy     : 1.15.3



#Disease Classification

##LSTM Model

In [ ]:
bal_all = np.loadtxt('/content/drive/MyDrive/ML/Disease_classification/Dataset/both_phases.csv', delimiter=',')

In [ ]:
x_bal = bal_all[:,0:1024]
y_disease = bal_all[:,-1]
print(np.shape(x_bal))
print(np.shape(y_disease))
y_disease=y_disease.reshape(-1,1 )
print(np.shape(y_disease))

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

print(y_disease[8500,0])
enc = OneHotEncoder(handle_unknown='ignore')
# 2. FIT
enc.fit(y_disease)

# 3. Transform
onehotlabels = enc.transform(y_disease).toarray()

print(onehotlabels[8500])

x_train1, x_test1, y_train1, y_test1 = train_test_split(x_bal, onehotlabels, test_size = 0.2, shuffle = True)

(29310, 1024)
(29310,)
(29310, 1)
1.0
[0. 1. 0. 0. 0.]


In [ ]:
import numpy as np
from keras.models import Sequential
from keras.layers import (Dense, LSTM, Conv1D, MaxPooling1D, BatchNormalization, Dropout)
from keras.callbacks import EarlyStopping
from keras.layers import Input

# Start timing
from time import time
start = time()

# Define the model
model = Sequential()

# Input layer
model.add(Input(shape=(1024, 1)))

# Convolutional layers with increased complexity
model.add(Conv1D(filters=64, kernel_size=7, strides=1, padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv1D(filters=64, kernel_size=5, strides=1, padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2, padding='same'))
model.add(Dropout(0.3))

model.add(Conv1D(filters=128, kernel_size=7, strides=1, padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv1D(filters=128, kernel_size=5, strides=1, padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2, padding='same'))
model.add(Dropout(0.3))

model.add(Conv1D(filters=256, kernel_size=3, strides=1, padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2, padding='same'))
model.add(Dropout(0.4))

# LSTM layers with increased units for capturing temporal patterns
model.add(LSTM(200, return_sequences=True, dropout=0.3, recurrent_dropout=0.3))
model.add(LSTM(100, return_sequences=False, dropout=0.3, recurrent_dropout=0.3))

# Fully connected layers for classification
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.4))

# Output layer
model.add(Dense(5, activation='softmax'))

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['CategoricalAccuracy'])

# Display model summary
print(model.summary())

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=10, verbose=2, restore_best_weights=True)

# Fit the model
history_m = model.fit(
    x_train1, y_train1,
    epochs=50,  # Increased epochs for complex training
    validation_split=0.2,
    callbacks=[early_stopping],
    batch_size=64,
    shuffle=True,
    verbose=1
)

# Total training time
t = time() - start
print('Total training time:', t)
history_m.history['training_time'] = t

# Evaluate the model
scores = model.evaluate(x_test1, y_test1, verbose=0)
print("Accuracy: {:.2f}%".format(scores[1] * 100))

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_7 (Conv1D)               │ (None, 1024, 64)       │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 1024, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 1024, 64)       │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 1024, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 512, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 512, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (None, 512, 128)       │        57,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 512, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_10 (Conv1D)              │ (None, 512, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 512, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_6 (MaxPooling1D)  │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ (None, 256, 256)       │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 256, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_7 (MaxPooling1D)  │ (None, 128, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 128, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128, 200)       │       365,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 100)            │       120,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        12,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 769,205 (2.93 MB)

 Trainable params: 767,925 (2.93 MB)

 Non-trainable params: 1,280 (5.00 KB)

None
Epoch 1/50
294/294 ━━━━━━━━━━━━━━━━━━━━ 1203s 4s/step - CategoricalAccuracy: 0.3358 - loss: 1.4568 - val_CategoricalAccuracy: 0.4399 - val_loss: 1.2422
Epoch 2/50
294/294 ━━━━━━━━━━━━━━━━━━━━ 1183s 4s/step - CategoricalAccuracy: 0.4609 - loss: 1.1887 - val_CategoricalAccuracy: 0.5330 - val_loss: 1.0434
Epoch 3/50
294/294 ━━━━━━━━━━━━━━━━━━━━ 1150s 4s/step - CategoricalAccuracy: 0.4995 - loss: 1.1068 - val_CategoricalAccuracy: 0.5145 - val_loss: 1.1508
Epoch 4/50
294/294 ━━━━━━━━━━━━━━━━━━━━ 1174s 4s/step - CategoricalAccuracy: 0.4703 - loss: 1.1934 - val_CategoricalAccuracy: 0.5252 - val_loss: 1.0566
Epoch 5/50
294/294 ━━━━━━━━━━━━━━━━━━━━ 1232s 4s/step - CategoricalAccuracy: 0.5396 - loss: 1.0284 - val_CategoricalAccuracy: 0.5249 - val_loss: 1.0807
Epoch 6/50
294/294 ━━━━━━━━━━━━━━━━━━━━ 1225s 4s/step - CategoricalAccuracy: 0.5203 - loss: 1.0566 - val_CategoricalAccuracy: 0.5017 - val_loss: 1.1146
Epoch 7/50
294/294 ━━━━━━━━━━━━━━━━━━━━ 1191s 4s/step - CategoricalAccuracy: 0.5511

KeyboardInterrupt: 